In [24]:
import importlib.util

spec = importlib.util.find_spec("pyadomd")
if spec and spec.origin:
    print(spec.origin)  # Path to pyadomd/__init__.py or main file
    print(spec.submodule_search_locations[0])  # Directory containing the package
else:
    print("pyadomd not found")

c:\Users\Cory.Cundy\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyadomd\__init__.py
c:\Users\Cory.Cundy\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyadomd


In [30]:
import sys
import json
import pandas as pd
import hashlib
import glob
import os
import shutil
import importlib.util
import filecmp

# Ensure the ADOMD.NET path is in the system path prior to importing Pyadomd
from sys import path
adomd_path = r'C:\Program Files\Microsoft.NET\ADOMD.NET\160'
if adomd_path not in path:
    path.append(adomd_path)

working_directory = os.getcwd()

spec = importlib.util.find_spec("pyadomd")
if spec and spec.origin and len(spec.submodule_search_locations) > 0:
    pyadomd_folder = spec.submodule_search_locations[0]
    src_file = "pyadomd.txt"  # Path to your source file (adjust if needed)
    # src_file = os.path.join(working_directory, "pyadomd.txt")
    dst_file = os.path.join(pyadomd_folder, "pyadomd.py")
    if not filecmp.cmp(src_file, dst_file, shallow=False):
        shutil.copy(src_file, dst_file)
else:
    print("pyadomd not found")
    sys.exit(1)

#TODO: Copy pyadomd.txt to overwrite the file in the ADOMD.NET folder
# C:\Users\Cory.Cundy\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyadomd

# Import Pyadomd after ensuring the path is set
from pyadomd import Pyadomd

def find_visual_id(id_map, start_id):
    current_id = start_id
    while current_id:
        node = id_map.get(current_id)
        if not node:
            break
        metrics = node.get('metrics', {})
        visual_id = metrics.get('visualId')
        if visual_id:
            return visual_id
        current_id = node.get('parentId')
    return None

# Flatten metrics into top-level columns
def flatten_event(event):
    flat = event.copy()
    metrics = flat.pop("metrics", {})
    flat.update(metrics)
    return flat

def row_hash(row):
    # Concatenate all values as string and hash
    row_str = '|'.join(str(val) for val in row.values)
    return hashlib.sha256(row_str.encode('utf-8')).hexdigest()

project_name = "Sales & Returns Sample"
pbi_pa_folder_name = "PBI PA Files"
baseline_folder_name = "baseline"
baseline_parquet_file_name = "baseline.parquet"
baseline_csv_file_name = "baseline.csv"
output_csv_file = ""
output_parquet_file = ""
instance_folder = ""
instance_csv_file = ""
instance_parquet_file = ""

is_baseline = False  # Set to True if this run is a baseline, False otherwise

# Connection string to your Power BI semantic model (update as needed)
connection_string = "Provider=MSOLAP;Data Source=localhost:52253;Initial Catalog=0089fece-2176-4d08-9a8f-2272c4cfe691"

project_folder = os.path.join(working_directory, project_name)


# Specify the folder containing your JSON files
pbi_pa_folder = os.path.join(project_folder, pbi_pa_folder_name)
#project_folder = rf"C:\Users\Cory.Cundy\source\repos\Power BI Regression Testing\{project_name}"
#baseline_folder = rf"{project_folder}\baseline"
baseline_folder = os.path.join(project_folder, baseline_folder_name)

#pbi_pa_folder = rf"{project_folder}\{pbi_pa_folder_name}"

#baseline_folder = os.path.join(os.getcwd(), rf"{project_name}\baseline")
#baseline_file = os.path.join(baseline_folder, "baseline.parquet")
baseline_csv_file = os.path.join(baseline_folder, baseline_csv_file_name)
baseline_parquet_file = os.path.join(baseline_folder, baseline_parquet_file_name)

# if os.path.isdir(baseline_folder):
#     if os.path.isfile(baseline_parquet_file):
#         baseline_exists = True
#         print("Baseline folder and baseline.parquet file exist.")
#     else:
#         baseline_exists = False
#         print("Baseline folder exists, but baseline.parquet file is missing.")
# elif os.path.isfile(baseline_parquet_file):
#     baseline_exists = True
#     print("Baseline.parquet file exists, but baseline folder is missing.")
# else:
#     baseline_exists = False

baseline_exists = os.path.isfile(baseline_parquet_file)
user_value = input("Press enter to create a baseline, otherwise enter a instance name: ")

if user_value.strip() == "":
    instance_name = baseline_folder_name
    is_baseline = True

    if baseline_exists:
        overwrite = input("Do you want to overwrite the baseline (Y/N)? ").strip().lower() 
        if overwrite != 'y':
            print("Exiting without creating a new baseline.")
            sys.exit()
    else:
        print("Creating a new baseline folder.")
        os.makedirs(baseline_folder, exist_ok=True)
        print("Baseline folder created.")
else:
    is_baseline = False
    if not baseline_exists:
        print("Baseline folder does not exist. Please create a baseline first.")
        sys.exit()
    else:
        instance_name = user_value.strip()
        instance_folder = os.path.join(project_folder, instance_name)
        instance_csv_file = os.path.join(instance_folder, f"{instance_name}.csv")
        instance_parquet_file = os.path.join(instance_folder, f"{instance_name}.parquet")
        instance_exists = os.path.isfile(instance_parquet_file)

        if instance_exists:
            overwrite = input("Do you want to overwrite the instance (Y/N)? ").strip().lower() 

            if overwrite != 'y':
                print("Exiting without creating a new instance.")
                sys.exit()
        else:
            os.makedirs(instance_folder, exist_ok=True)

if is_baseline:
    output_csv_file = baseline_csv_file
    output_parquet_file = baseline_parquet_file
else:
    output_csv_file = instance_csv_file
    output_parquet_file = instance_parquet_file

all_dfs = []

for json_path in glob.glob(os.path.join(pbi_pa_folder, r"*.json")):
    with open(json_path, "r", encoding="utf-8-sig") as f:
        data = json.load(f)
    events = data.get("events", [])
    id_map = {event['id']: event for event in events if 'id' in event}
    flat_events = [flatten_event(e) for e in events]
    df = pd.DataFrame(flat_events)
    if not df.empty:
        df['visualId'] = df['id'].apply(lambda x: find_visual_id(id_map, x))
        all_dfs.append(df)

# Combine all DataFrames into one
if all_dfs:
    final_df = pd.concat(all_dfs, ignore_index=True)
else:
    final_df = pd.DataFrame()

pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', 20)    

filtered_df = final_df[final_df['name'] == 'Execute DAX Query'].copy()
filtered_df.drop_duplicates(inplace=True)
filtered_df['result_sets'] = 0
filtered_df['combined_query_hash'] = ""

# Loop through the DataFrame and execute each DAX query
with Pyadomd(connection_string) as conn:
    #Loop over each query from the dataframe
    for idx, row in filtered_df.iterrows():
        query_id = row['id']
        query_text = row['QueryText']

        try:
            with conn.cursor().execute(query_text) as cur:
                result_set_index = 0
                query_hashes = []  # List to collect all query hashes
                has_next = True
                while has_next:
                    result_set_index += 1
                    columns = [col[0] for col in cur.description]
                    result_rows = cur.fetchall()
                    print(f"QueryID: {query_id} Result set {result_set_index} has {len(result_rows)} rows")

                    if result_rows:
                        # Create a DataFrame for this query's results
                        result_df = pd.DataFrame(result_rows, columns=columns)

                        # Add a hash column for each row
                        if not result_df.empty:
                            result_df['row_hash'] = result_df.apply(row_hash, axis=1)

                            # Sort by row_hash
                            result_df = result_df.sort_values('row_hash').reset_index(drop=True)
                            # Concatenate all row_hash values and hash them
                            row_hashes = '|'.join(result_df['row_hash'].tolist())
                            combined_row_hash = hashlib.sha256(row_hashes.encode('utf-8')).hexdigest()
                            query_hashes.append(combined_row_hash)

                            #print(f"query: {query_id} result set {result_set_index} rows: {len(result_df)} combined hash: {combined_row_hash}")
                        else:
                            print(f"query: {query_id} result set {result_set_index} is empty")

                    has_next = cur.nextresult()

                filtered_df.loc[idx, 'result_sets'] = result_set_index

                if query_hashes:
                    combined = '|'.join(query_hashes)
                    combined_query_hash = hashlib.sha256(combined.encode('utf-8')).hexdigest()
                    filtered_df.loc[idx, 'combined_query_hash'] = combined_query_hash
                else:
                    filtered_df.loc[idx, 'combined_query_hash'] = None
                    print(f"No result sets for query {query_id}")

        except Exception as e:
            print(f"Error executing query {query_id}: {e}\n")

# Drop NaN values and convert to string (in case)
all_hashes = filtered_df['combined_query_hash'].dropna().astype(str).tolist()
combined = '|'.join(all_hashes)
final_overall_hash = hashlib.sha256(combined.encode('utf-8')).hexdigest()
filtered_df['final_overall_hash'] = final_overall_hash

print(f"Final overall hash for all combined_query_hash values: {final_overall_hash}")

filtered_df = filtered_df[['id', 'parentId', 'visualId', 'QueryText', 'RowCount', 'result_sets', 'combined_query_hash', 'final_overall_hash']]

filtered_df.to_csv(output_csv_file, index=False)
filtered_df.to_parquet(output_parquet_file, index=False)

#If this run is not a baseline, then compare to the baseline
if baseline_exists and instance_name != baseline_folder_name:
    if not os.path.isfile(baseline_parquet_file):
        print("Baseline file does not exist. Please create a baseline first.")
        exit()

    baseline_df = pd.read_parquet(baseline_parquet_file)

    # Compare the current filtered_df with the baseline_df
    comparison_df = filtered_df.merge(baseline_df, on='id', suffixes=('', '_baseline'), how='outer', indicator=True)

    # Identify changes
    changes = comparison_df[comparison_df['_merge'] != 'both']
    if not changes.empty:
        print("Changes detected compared to the baseline:")
        print(changes)
    else:
        print("No changes detected compared to the baseline.")


    # Identify value differences in columns for rows present in both
    cols_to_compare = ['combined_query_hash']
    diff_mask = (comparison_df['_merge'] == 'both') & (
        comparison_df[[col for col in cols_to_compare]].ne(
            comparison_df[[f"{col}_baseline" for col in cols_to_compare]].values
        ).any(axis=1)
    )
    value_diffs = comparison_df[diff_mask]
    if not value_diffs.empty:
        print("Rows with differing values between current and baseline:")
        print(value_diffs[['id'] + [col for col in cols_to_compare] + [f"{col}_baseline" for col in cols_to_compare]])
    else:
        print("No value differences in shared rows.")

    display(value_diffs)


QueryID: 2bd3d61f-1ea4-4cb1-aaba-1743fea9a057 Result set 1 has 1 rows
QueryID: 2bd3d61f-1ea4-4cb1-aaba-1743fea9a057 Result set 2 has 1 rows
QueryID: b1f1c889-6489-4d11-9306-2478268354f5 Result set 1 has 1 rows
QueryID: be5a6d64-78e8-41f5-baf0-65e368cd1ff5 Result set 1 has 1 rows
QueryID: f220b8c6-968c-4be7-a824-8681380b5902 Result set 1 has 1 rows
QueryID: f622efd8-0128-43cf-bf15-26be4c4d4bce Result set 1 has 1 rows
QueryID: cc438199-3710-4961-a236-976791bd5a93 Result set 1 has 1 rows
QueryID: cf2e909a-b487-4116-a882-6ae749700b5d Result set 1 has 1 rows
QueryID: 7e7556c9-0e6d-4e29-a7f3-aa198a896354 Result set 1 has 8 rows
QueryID: 7bac1191-a040-470d-8dc1-456bc8693dc5 Result set 1 has 1 rows
QueryID: c553c914-4ce3-4aaf-9ff2-70766f89d1b1 Result set 1 has 1 rows
QueryID: b2099692-519a-4dea-9941-888c3d77edf9 Result set 1 has 1 rows
QueryID: 15be3836-c2a4-4292-bb1a-2b1e0e997d67 Result set 1 has 1 rows
QueryID: 7321fcdf-8038-4ca6-bbe1-db6ed61b32ed Result set 1 has 1 rows
QueryID: c522e7a5-62

,id,parentId,visualId,QueryText,RowCount,result_sets,combined_query_hash,final_overall_hash,parentId_baseline,visualId_baseline,QueryText_baseline,RowCount_baseline,result_sets_baseline,combined_query_hash_baseline,final_overall_hash_baseline,_merge
12,3116d681-710e-4c4a-b34b-aa867a01d347,5466294c-f42e-4373-9c22-0dc27418be93,b33397810d555ca70a8c,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tFIL...,16.0,2,a8db92e37a449349fc27882113c44b965f332dad142efb...,44edf9f657b65f9e8f68013332e2695053f87f39ed6e01...,5466294c-f42e-4373-9c22-0dc27418be93,b33397810d555ca70a8c,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tFIL...,16.0,2,113805e845ab8d302bf3a430f549c710267e1bf81c3578...,b21af9321776ba8db715cb1ae33094c428ce635e51dba8...,both
24,80d1289d-9fde-4a2a-83f7-7a1e6abfd856,69f38537-2166-447c-b254-f865c96bc6ff,805719ca6000cb000be2,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tFIL...,27.0,1,1e7f3d5bd1fae11ff21363539c2c3fd4bc979836134ece...,44edf9f657b65f9e8f68013332e2695053f87f39ed6e01...,69f38537-2166-447c-b254-f865c96bc6ff,805719ca6000cb000be2,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tFIL...,27.0,1,9d249633ca25d0a143eedc3e988b0242502ca1d6e1b021...,b21af9321776ba8db715cb1ae33094c428ce635e51dba8...,both
39,c8f382ac-e545-4879-bd35-67f20096b8d7,8e712962-ac31-485e-8c04-c0d576955b18,948988db07a4db09d58d,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tFIL...,26.0,1,9bcaf8e5411523b21dd4e096d41eee979d58c5d171c28b...,44edf9f657b65f9e8f68013332e2695053f87f39ed6e01...,8e712962-ac31-485e-8c04-c0d576955b18,948988db07a4db09d58d,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tFIL...,26.0,1,0fea39528cc2f3bfc39d83d0a11d892a376f194ade4cf4...,b21af9321776ba8db715cb1ae33094c428ce635e51dba8...,both
46,e033aaf7-a2aa-413f-8a2f-969a4deb4615,d96a4290-a4a6-4b4e-9ba8-7da7a725cb34,e5b7b6b90231b41809a1,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tFIL...,1.0,1,4eaca420673e3994a1748aa6bae951e616ad9d4a91861e...,44edf9f657b65f9e8f68013332e2695053f87f39ed6e01...,d96a4290-a4a6-4b4e-9ba8-7da7a725cb34,e5b7b6b90231b41809a1,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tFIL...,1.0,1,a3e30f2f2b65e66d1016efa882c59db7b1b8cf50503758...,b21af9321776ba8db715cb1ae33094c428ce635e51dba8...,both
